# Click prompts collector — whip videos

Sits on top of the SurgSAM-2 pipeline. For each whip video in `FRAMES_ROOT`:
1. The collector picks **3 prompt frames** — frame 0, N/3, 2·N/3 — so the model gets re-anchored mid-video.
2. On each prompt frame you click instruments:
   - **Left click** = positive (this pixel IS the instrument)
   - **Right click** = negative (this pixel is NOT — use to push the mask off confusing tissue)
   - Press **`1`** / **`2`** / **`3`** to switch which object you are clicking for. *Same number across frames = same physical instrument re-anchored.*
3. When done with a frame, close the figure window to advance.
4. After all 3 prompt frames are done, save the JSON.

Output: `prompts/<video>.json` — consumed by `run_on_video.py --prompts-json ...`.

**Requires** the `ipympl` backend (`%matplotlib widget`). Should already be installed in the venv.

In [ ]:
%matplotlib widget
from pathlib import Path
import json, os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Bigpurple path (frames live here, no download needed)
FRAMES_ROOT = Path('/gpfs/data/oermannlab/private_data/whip/frames_attempt2')
# Local fallback for testing on olab-1
if not FRAMES_ROOT.exists():
    FRAMES_ROOT = Path('test_data')

PROMPTS_DIR = Path('prompts')
PROMPTS_DIR.mkdir(exist_ok=True)

N_PROMPT_FRAMES = 3      # how many time-points to seed at

# Marker colours per obj_id (cycles after 8).
OBJ_COLORS = ['#00ff00', '#ff8800', '#00bfff', '#ff00ff', '#ffff00',
              '#ff0000', '#00ffff', '#ffffff']

def list_videos(root):
    out = []
    for d in sorted(root.iterdir()):
        if not d.is_dir(): continue
        if d.name.startswith('480full'): continue
        n = sum(1 for _ in d.glob('*.png')) + sum(1 for _ in d.glob('*.jpg'))
        if n < 3: continue
        out.append((d.name, n))
    return out

videos = list_videos(FRAMES_ROOT)
print(f'Found {len(videos)} videos under {FRAMES_ROOT}')
for v, n in videos[:5]:
    print(f'  {v}: {n} frames')
if len(videos) > 5:
    print(f'  ... and {len(videos)-5} more')

In [ ]:
def get_sorted_frames(video_dir):
    """Return frames sorted by their numeric component."""
    import re
    rx = re.compile(r'(\d+)')
    items = []
    for p in video_dir.iterdir():
        if p.suffix.lower() not in {'.png', '.jpg', '.jpeg'}: continue
        m = rx.search(p.stem)
        if not m: continue
        items.append((int(m.group(1)), p))
    items.sort()
    return [p for _, p in items]


def collect_clicks_for_frame(image_path, title):
    """Open one figure, collect clicks until the user closes it.

    Returns dict: {obj_id: {'positive': [[x,y],...], 'negative': [[x,y],...]}}"""
    img = Image.open(image_path)
    fig, ax = plt.subplots(figsize=(13, 7.5))
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    fig.tight_layout()

    obj_id_state = {'cur': 1}
    result = {}
    info_text = ax.text(0.01, 0.99, f'obj 1  (Left=+, Right=-, press 1/2/3 to switch, close window when done)',
                        transform=ax.transAxes, ha='left', va='top',
                        fontsize=11, color='white',
                        bbox=dict(facecolor='black', alpha=0.6, edgecolor='none'))

    def refresh_info():
        info_text.set_text(f'obj {obj_id_state["cur"]}  '
                           f'(Left=+, Right=-, press 1/2/3 to switch, close window when done)')
        info_text.set_bbox(dict(facecolor=OBJ_COLORS[(obj_id_state['cur']-1) % len(OBJ_COLORS)],
                                alpha=0.85, edgecolor='none'))

    def on_click(event):
        if event.inaxes != ax or event.xdata is None: return
        oid = obj_id_state['cur']
        result.setdefault(oid, {'positive': [], 'negative': []})
        color = OBJ_COLORS[(oid-1) % len(OBJ_COLORS)]
        if event.button == 1:
            result[oid]['positive'].append([round(event.xdata, 1), round(event.ydata, 1)])
            ax.plot(event.xdata, event.ydata, marker='+', color=color,
                    markersize=22, markeredgewidth=3)
        elif event.button == 3:
            result[oid]['negative'].append([round(event.xdata, 1), round(event.ydata, 1)])
            ax.plot(event.xdata, event.ydata, marker='x', color=color,
                    markersize=22, markeredgewidth=3)
        fig.canvas.draw_idle()

    def on_key(event):
        if event.key in '123456789':
            obj_id_state['cur'] = int(event.key)
            refresh_info()
            fig.canvas.draw_idle()

    refresh_info()
    fig.canvas.mpl_connect('button_press_event', on_click)
    fig.canvas.mpl_connect('key_press_event', on_key)
    plt.show()
    return result


def collect_for_video(video_name, n_prompt_frames=N_PROMPT_FRAMES, save=True):
    """Walk the 3 prompt frames for a video, collect clicks, save JSON."""
    video_dir = FRAMES_ROOT / video_name
    frames = get_sorted_frames(video_dir)
    if not frames:
        print(f'No frames in {video_dir}')
        return None
    n_total = len(frames)
    # uniformly spaced prompt frame indices
    idxs = np.linspace(0, n_total-1, n_prompt_frames, dtype=int)

    # We index by output-frame index, which is loader_dir's sequential index (0..n_total-1)
    # — same as what the runner sees. Source-file names are not what the model addresses.
    objects_by_frame = {}
    first_img = Image.open(frames[0])
    W, H = first_img.size
    print(f'{video_name}: {n_total} frames at {W}x{H}; will prompt at indices {list(idxs)}')

    for i, fidx in enumerate(idxs):
        title = (f'{video_name} — prompt frame {i+1}/{n_prompt_frames}  '
                 f'(loader idx {fidx} = source {frames[fidx].stem})\n'
                 f'Click instruments. Same obj_id across frames = same instrument.')
        clicks = collect_clicks_for_frame(frames[fidx], title)
        if clicks:
            objects_by_frame[int(fidx)] = [
                {'obj_id': oid, 'positive': v['positive'], 'negative': v['negative']}
                for oid, v in sorted(clicks.items())
            ]
        else:
            print(f'  (no clicks recorded at frame {fidx})')

    out = {
        'video': video_name,
        'resolution': [W, H],
        'n_frames': n_total,
        'prompt_frames': [int(i) for i in idxs],
        'objects_by_frame': {str(k): v for k, v in objects_by_frame.items()},
    }
    if save:
        out_path = PROMPTS_DIR / f'{video_name}.json'
        with open(out_path, 'w') as fp:
            json.dump(out, fp, indent=2)
        print(f'  saved -> {out_path}')
    return out

print('Helpers loaded. Run the next cell to start clicking on one video at a time.')

## Click on one video at a time

Pick the video name from the list above. Run the cell; a figure opens for prompt frame 1. Click instruments, switch object id with number keys, close the figure to advance. Three figures in series, then the JSON is saved.

In [ ]:
VIDEO = 'DG_whip_16598313'    # <-- change this for each new video
out = collect_for_video(VIDEO)
print('Summary:')
print(json.dumps(out, indent=2))

## Walk every un-done video (alternative to one-at-a-time)

Loops through all videos that don't yet have a `prompts/<video>.json`. Set `LIMIT` to cap how many you tackle in one sitting.

In [ ]:
LIMIT = 5  # set to None for all of them
done = 0
for vname, _ in videos:
    target = PROMPTS_DIR / f'{vname}.json'
    if target.exists():
        continue
    collect_for_video(vname)
    done += 1
    if LIMIT is not None and done >= LIMIT:
        break
print(f'Did {done} videos this session. {sum(1 for v,_ in videos if (PROMPTS_DIR/(v+".json")).exists())} / {len(videos)} total done.')